In [ ]:
#!/usr/bin/env python3
# pip install evaluate sacrebleu rouge-score nltk
# pip install rouge-score sacrebleu nltk evaluate
import os
import time
import asyncio
import aiohttp
import pandas as pd
import nltk
import evaluate
import nest_asyncio
from rouge_score import rouge_scorer as rs_lib
import sacrebleu
from nltk.translate.meteor_score import meteor_score as meteor_fn
from nltk.tokenize import word_tokenize
import nltk
from collections import Counter
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nest_asyncio.apply()

API_KEY         = ""
OPENROUTER_URL  = "https://openrouter.ai/api/v1/chat/completions"
CSV_PATH        = "HealthParaphrasing_stratified_200.csv"
MODEL           = "openai/gpt-5-mini"
OUTPUT_DIR      = "/content"
NUM_SAMPLES     = 200
CONCURRENCY     = 10
BATCH_SIZE      = 10
BATCH_DELAY     = 2
SITE_URL        = "https://colab.research.google.com"
SITE_NAME       = "Bangla Paraphrase Benchmark"
MAX_RETRIES     = 1
BASE_BACKOFF    = 2.0
REQUEST_TIMEOUT = 60

REASONING_EFFORT = "none"   # set to "low"/"medium"/"high" to enable reasoning

THREE_SHOT_EXAMPLES = [
    {
        "source":    "আর বিভিন্ন খাবারের মাধ্যমে এই যোগটি শরীরে উৎপাদিত হয়",
        "paraphrase": "আর এই যোগ শরীরে তৈরি হয় বিভিন্ন খাবারের মাধ্যমে।"
    },
    {
        "source":    "মানুষের বুকের বাম এবং ডান দিকে দুটি ফুসফুস আছে",
        "paraphrase": "মানুষের বুকের বাম ও ডান পাশে দুটি ফুসফুস থাকে।"
    },
    {
        "source":    "রোগীর ইতিহাস ও শারীরিক পরীক্ষার পর ইমেজিং সবচেয়ে কার্যকর পরীক্ষা যেমন এক্সরে প্রয়োজনে সিটি স্ক্যান এমআরআই পেট স্ক্যান বোন স্ক্যান",
        "paraphrase": "রোগীর ইতিহাস এবং শারীরিক পরীক্ষার পরে, ইমেজিং হল সবচেয়ে দরকারী পরীক্ষা যেমন সিটি স্ক্যান, এমআরআই, পেট স্ক্যান, প্রয়োজনে হাড় স্ক্যান।"
    },
]

SYSTEM_PROMPT = """You are a Bengali paraphrase generation model.
Given a Bengali sentence, generate a single paraphrased version of it.
Rules:
- Preserve the original meaning exactly.
- Vary the vocabulary and sentence structure.
- Output ONLY the paraphrased Bengali sentence, nothing else.
- Do not include explanations, labels, or any extra text."""


def build_few_shot_messages(sentence: str) -> list:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in THREE_SHOT_EXAMPLES:
        messages.append({"role": "user",      "content": f"Paraphrase this Bengali sentence:\n{ex['source']}"})
        messages.append({"role": "assistant", "content": ex["paraphrase"]})
    messages.append({"role": "user", "content": f"Paraphrase this Bengali sentence:\n{sentence}"})
    return messages


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def _rouge_n(pred_tokens, ref_tokens, n):
    pred_ng = _ngrams(pred_tokens, n)
    ref_ng  = _ngrams(ref_tokens,  n)
    overlap = sum((pred_ng & ref_ng).values())
    p = overlap / max(sum(pred_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()),  1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f


def _lcs_len(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def _rouge_l(pred_tokens, ref_tokens):
    lcs = _lcs_len(pred_tokens, ref_tokens)
    p = lcs / max(len(pred_tokens), 1)
    r = lcs / max(len(ref_tokens),  1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f


def build_headers():
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
        "HTTP-Referer":  SITE_URL,
        "X-Title":       SITE_NAME,
    }


async def call_model_async(session, model, messages, headers):
    reasoning_on = REASONING_EFFORT != "none"
    payload = {
        "model": model,
        "messages": messages,
        "max_completion_tokens": 1500,
        "reasoning_effort": "minimal",
        "temperature": 0.3,
    }
    start = time.monotonic()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with session.post(
                OPENROUTER_URL, headers=headers, json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
            ) as resp:
                if resp.status == 429:
                    wait = float(resp.headers.get("Retry-After", BASE_BACKOFF * attempt))
                    await asyncio.sleep(wait)
                    continue
                if resp.status >= 500:
                    await asyncio.sleep(BASE_BACKOFF * attempt)
                    continue
                if resp.status != 200:
                    text = await resp.text()
                    return None, f"http_error:{resp.status}:{text[:200]}", time.monotonic() - start, {}, reasoning_on
                data    = await resp.json()
                content = data["choices"][0]["message"]["content"].strip()

                usage          = data.get("usage", {})
                reasoning_tokens = usage.get("completion_tokens_details", {}).get("reasoning_tokens", 0)
                total_tokens     = usage.get("total_tokens", 0)

                token_info = {
                    "reasoning_on":      reasoning_on,
                    "reasoning_tokens":  reasoning_tokens,
                    "total_tokens":      total_tokens,
                }
                return content, None, time.monotonic() - start, token_info, reasoning_on
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start, {}, reasoning_on
            await asyncio.sleep(BASE_BACKOFF * attempt)
    return None, "max_retries_exceeded", time.monotonic() - start, {}, reasoning_on


def checkpoint_path(model):
    # checkpoint always goes to /content, named without sample-size suffix
    safe_model = model.replace('/', '__')
    return f"/content/checkpoint_{safe_model}.csv"


def load_checkpoint(model):
    path = checkpoint_path(model)
    return pd.read_csv(path) if os.path.exists(path) else None


def save_checkpoint(model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(model), index=False)


async def run_model_async(model, df, headers):
    existing  = load_checkpoint(model)
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    reasoning_on = REASONING_EFFORT != "none"
    print(f"\n── Reasoning: {'ON  (effort={})'.format(REASONING_EFFORT) if reasoning_on else 'OFF'} ──")

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows_done

    sem     = asyncio.Semaphore(CONCURRENCY)
    indices = list(range(start_idx, len(df)))
    batches = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]
    results = []

    async def process_row(session, row_idx):
        async with sem:
            row       = df.iloc[row_idx]
            source    = str(row["source_sentence"])
            reference = str(row["paraphrased_sentence"])

            messages = build_few_shot_messages(source)
            prediction, err, latency, token_info, _ = await call_model_async(session, model, messages, headers)

            if prediction is None:
                prediction = ""

            reasoning_tok = token_info.get("reasoning_tokens", 0)
            total_tok     = token_info.get("total_tokens", 0)
            print(
                f"[{row_idx}] err={err} | "
                f"reasoning={'ON' if reasoning_on else 'OFF'} | "
                f"reasoning_tokens={reasoning_tok} | "
                f"total_tokens={total_tok} | "
                f"src={source[:40]}..."
            )

            return {
                "_idx":             row_idx,
                "source":           source,
                "reference":        reference,
                "prediction":       prediction if not err else "",
                "latency":          latency,
                "error":            err,
                "reasoning_on":     reasoning_on,
                "reasoning_tokens": reasoning_tok,
                "total_tokens":     total_tok,
            }

    async with aiohttp.ClientSession() as session:
        for b_num, batch in enumerate(batches):
            print(f"\nBatch {b_num + 1}/{len(batches)} — rows {batch[0]}..{batch[-1]}")
            tasks         = [process_row(session, i) for i in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)

            all_rows = rows_done + sorted(results, key=lambda r: r["_idx"])
            save_checkpoint(model, all_rows)

            if b_num < len(batches) - 1:
                print(f"Waiting {BATCH_DELAY}s…")
                await asyncio.sleep(BATCH_DELAY)

    results = sorted(results, key=lambda r: r["_idx"])
    for r in results:
        del r["_idx"]
    all_rows = rows_done + results
    save_checkpoint(model, all_rows)
    return all_rows


def run_model_on_dataset(model, df, headers):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_model_async(model, df, headers))


def compute_metrics(rows):
    predictions, references, latencies = [], [], []
    total_reasoning_tokens, total_tokens_all = 0, 0

    for r in rows:
        pred = str(r.get("prediction", "")).strip()
        ref  = str(r.get("reference",  "")).strip()
        err  = r.get("error")
        if pred and ref and (err is None or str(err).strip() == ""):
            predictions.append(pred)
            references.append(ref)
        if r.get("latency") is not None:
            latencies.append(float(r["latency"]))
        total_reasoning_tokens += int(r.get("reasoning_tokens", 0) or 0)
        total_tokens_all       += int(r.get("total_tokens",     0) or 0)

    if not predictions:
        print("No valid predictions found.")
        return {}

    print(f"Evaluating {len(predictions)} valid predictions...")

    bleu_score = sacrebleu.corpus_bleu(
        predictions, [references], tokenize="none"
    ).score / 100

    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        pt = pred.split()
        rt = ref.split()
        r1.append(_rouge_n(pt, rt, 1))
        r2.append(_rouge_n(pt, rt, 2))
        rl.append(_rouge_l(pt, rt))

    meteor_scores = [
        meteor_fn([ref.split()], pred.split())
        for pred, ref in zip(predictions, references)
    ]

    return {
        "BLEU":                  bleu_score,
        "ROUGE-1 (F1)":          sum(r1) / len(r1),
        "ROUGE-2 (F1)":          sum(r2) / len(r2),
        "ROUGE-L (F1)":          sum(rl) / len(rl),
        "METEOR":                sum(meteor_scores) / len(meteor_scores),
        "total":                 len(predictions),
        "avg_latency":           sum(latencies) / len(latencies) if latencies else None,
        "total_reasoning_tokens": total_reasoning_tokens,
        "total_tokens_all":      total_tokens_all,
    }


def safe_model_name(model):
    return model.replace("/", "__").replace(":", "-")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    if not {"source_sentence", "paraphrased_sentence"}.issubset(df.columns):
        raise ValueError("CSV must contain 'source_sentence' and 'paraphrased_sentence' columns")

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    sample_size = len(df)
    print(f"\nDataset: {sample_size} sentence pairs")
    print(f"Prompting: 3-shot")

    reasoning_on = REASONING_EFFORT != "none"
    print(f"Reasoning: {'ON  (effort=' + REASONING_EFFORT + ')' if reasoning_on else 'OFF'}")

    headers    = build_headers()
    rows       = run_model_on_dataset(MODEL, df, headers)
    metrics    = compute_metrics(rows)

    model_tag  = safe_model_name(MODEL)
    base_name  = f"{model_tag}_{sample_size}"

    pred_path    = os.path.join(OUTPUT_DIR, f"{base_name}_predictions.csv")
    results_path = os.path.join(OUTPUT_DIR, f"{base_name}_benchmarkResults.csv")

    pd.DataFrame(rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame([{
        "model":                  MODEL,
        "prompting":              "3-shot",
        "reasoning":              "ON" if reasoning_on else "OFF",
        "reasoning_effort":       REASONING_EFFORT,
        "sample_size":            sample_size,
        "BLEU":                   round(metrics["BLEU"], 4),
        "ROUGE-1 (F1)":           round(metrics["ROUGE-1 (F1)"], 4),
        "ROUGE-2 (F1)":           round(metrics["ROUGE-2 (F1)"], 4),
        "ROUGE-L (F1)":           round(metrics["ROUGE-L (F1)"], 4),
        "METEOR":                 round(metrics["METEOR"], 4),
        "total_sentences":        metrics["total"],
        "avg_latency_s":          metrics["avg_latency"],
        "total_reasoning_tokens": metrics["total_reasoning_tokens"],
        "total_tokens_all":       metrics["total_tokens_all"],
    }])
    results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

    print(f"\nPredictions saved : {pred_path}")
    print(f"Results saved     : {results_path}")

    print("\n── Evaluation Results (paper Table 5 format) ──")
    paper_scores = {
        "BLEU": 0.1919, "ROUGE-1 (F1)": 0.5160,
        "ROUGE-2 (F1)": 0.2761, "ROUGE-L (F1)": 0.4903, "METEOR": 0.4777
    }
    for metric in ["BLEU", "ROUGE-1 (F1)", "ROUGE-2 (F1)", "ROUGE-L (F1)", "METEOR"]:
        your  = metrics[metric]
        paper = paper_scores[metric]
        diff  = your - paper
        sign  = "+" if diff >= 0 else ""
        print(f"  {metric:<15} yours={your:.4f}  paper={paper:.4f}  diff={sign}{diff:.4f}")

    print(f"\n  Total evaluated       : {metrics['total']} sentences")
    print(f"  Avg latency           : {metrics['avg_latency']:.2f}s" if metrics['avg_latency'] else "")
    print(f"  Reasoning             : {'ON  (effort=' + REASONING_EFFORT + ')' if reasoning_on else 'OFF'}")
    print(f"  Total reasoning tokens: {metrics['total_reasoning_tokens']}")
    print(f"  Total tokens (all)    : {metrics['total_tokens_all']}")


if __name__ == "__main__":
    main()


Dataset: 200 sentence pairs
Prompting: 3-shot
Reasoning: OFF

── Reasoning: OFF ──

Batch 1/20 — rows 0..9
[0] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=338 | src=আর বিভিন্ন খাবারের মাধ্যমে এই যোগটি শরীর...
[4] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=351 | src=তখন জ্বর কাশি হাঁচি শ্বাসকষ্ট নিউমোনিয়া ...
[8] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=346 | src= যাদের ডায়াবেটিস আছে তাদের ১০ শতাংশ এই ...
[3] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=362 | src=আর্থ্রাইটিস যে কারণের জন্য হয়েছে সে কারণ...
[2] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=343 | src=মানুষের বুকের বাম এবং ডান দিকে দুটি ফুসফ...
[5] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=367 | src=এরকম বাঁধার সৃষ্টি হতে পারে যদি জন্মগতভা...
[6] err=None | reasoning=OFF | reasoning_tokens=0 | total_tokens=377 | src=দিনের পর দিন এ ধরনের খাবার খেলে ওজন যেমন...
[9] err=None | reasoning=OFF | reasoning_tokens=0 | total_t